# 3.2 LGS lösen mit NumPy

Im letzten Kapitel haben wir das Obstmarkt-Problem als Matrixgleichung
$\mathbf{A} \cdot \vec{x} = \vec{b}$ aufgeschrieben und mit der Determinante
geprüft, ob eine eindeutige Lösung existiert. Jetzt berechnen wir sie.

## Lernziele

* [ ] Sie können ein LGS mit `np.linalg.solve(A, b)` lösen.
* [ ] Sie können das Ergebnis mit einer Probe absichern.
* [ ] Sie wissen, wofür `np.linalg.inv()` sinnvoll ist und wann `solve` vorzuziehen ist.

## Lösen mit `np.linalg.solve`

Wir starten sofort mit dem Code. Das Obstmarkt-System aus Kapitel 3.1 kennen
wir bereits:

In [ ]:
import numpy as np

# Koeffizientenmatrix: Zeile = Einkaufstag, Spalte = Fruchtsorte
A = np.array([
    [3, 2, 1],
    [2, 3, 0],
    [1, 1, 3],
], dtype=float)

# Rechte Seite: bezahlte Gesamtbeträge in Euro
b = np.array([2.10, 1.70, 2.80])

# np.linalg.solve löst A * x = b in einer einzigen Zeile
# Intern verwendet NumPy eine LU-Zerlegung (schneller als Gauß von Hand)
# Wichtig: solve erwartet immer zuerst A, dann b
x = np.linalg.solve(A, b)

# x ist ein 1D-Array: x[0]=Apfel, x[1]=Banane, x[2]=Clementinen
print(f'Preis Apfel:      {x[0]:.2f} €')
print(f'Preis Banane:     {x[1]:.2f} €')
print(f'Preis Clementine: {x[2]:.2f} €')

## Das Ergebnis immer prüfen: die Probe

*Woher wissen wir, dass das Ergebnis stimmt?* Wenn $\vec{x}$ die richtige
Lösung ist, muss das Matrixprodukt $\mathbf{A} \cdot \vec{x}$ exakt den
Vektor $\vec{b}$ reproduzieren. Das ist unsere **Probe**:

In [ ]:
# Probe: A mal x muss gleich b ergeben
# @ ist der Matrixmultiplikations-Operator in NumPy (nicht * verwenden!)
b_probe = A @ x

print('Ergebnis A @ x:', b_probe)
print('Soll-Wert   b: ', b)

# np.allclose prüft, ob alle Einträge bis auf winzige Rundungsfehler gleich sind
# Hintergrund: Fließkommazahlen sind nie exakt, daher schlägt == oft fehl
# allclose toleriert Abweichungen kleiner als 1e-8 (Standard-Schwellenwert)
print('Probe bestanden:', np.allclose(b_probe, b))

Eine einfache Zuweisung `B = A` erstellt **keine** unabhängige Kopie, sondern
nur einen zweiten Namen für dasselbe Array im Speicher. Änderungen an `B`
verändern dann auch `A`, ohne jede Fehlermeldung. Um eine echte, unabhängige
Kopie zu erhalten, verwenden Sie immer `B = A.copy()`.

### Mini-Übung 1

Legen Sie mit `A_falsch = A.copy()` eine Kopie von `A` an und ersetzen Sie
darin `A_falsch[0, 0]` durch `4` statt `3`. Lösen Sie das System mit
`A_falsch` und führen Sie die Probe durch. Was gibt
`np.allclose(A_falsch @ x_falsch, b)` aus und was bedeutet das für die
Verlässlichkeit der Probe?

*Hinweis:* Verwenden Sie `.copy()`, denn ohne diese Methode würden Sie
versehentlich auch das Original-Array `A` verändern.

In [ ]:
# Hier Ihren Code eingeben

## Die Inverse: eine Alternative für Spezialfälle (optional)

Dieser Abschnitt ist für das Verständnis von Kapitel 3.3 nicht erforderlich.
Er lohnt sich, wenn Sie verstehen möchten, wann `np.linalg.inv` gegenüber
`np.linalg.solve` sinnvoll ist.

Es gibt einen zweiten Weg zur Lösung. Aus
$\mathbf{A} \cdot \vec{x} = \vec{b}$ folgt durch Multiplikation mit der
**inversen Matrix** $\mathbf{A}^{-1}$ von links:

$$\vec{x} = \mathbf{A}^{-1} \cdot \vec{b}$$

Die Inverse ist so definiert, dass $\mathbf{A}^{-1} \cdot \mathbf{A}$
die **Einheitsmatrix** $\mathbf{I}$ ergibt: Einsen auf der Diagonale,
Nullen überall sonst. Das ist das Matrix-Äquivalent der Aussage
$\frac{1}{a} \cdot a = 1$:

In [ ]:
# Inverse berechnen
A_inv = np.linalg.inv(A)

# Probe: A_inv @ A soll die Einheitsmatrix ergeben
# np.round auf 10 Stellen, damit winzige Rundungsfehler nicht stören
print('A_inv @ A:')
print(np.round(A_inv @ A, 10))

# Lösung über die Inverse: dieselbe wie mit solve
x_via_inv = A_inv @ b
print('Lösung via Inverse:', np.round(x_via_inv, 2))

**Zum Lösen von LGS immer `np.linalg.solve`.** Es ist schneller und
numerisch stabiler als der Umweg über die Inverse, weil es die Matrix nicht
explizit invertiert. Das gilt auch dann, wenn viele verschiedene rechte
Seiten $\vec{b}_1, \vec{b}_2, \ldots$ vorliegen: `solve(A, B)` akzeptiert
eine Matrix `B`, deren Spalten die verschiedenen rechten Seiten sind, und
löst alle in einem einzigen Aufruf.

**`np.linalg.inv`** ist kein Lösungsverfahren für LGS, sondern berechnet
die Inverse als Ergebnis. Sinnvoll ist es nur, wenn $\mathbf{A}^{-1}$
selbst weiterverwendet werden soll, etwa in einer analytischen Herleitung
oder beim Prüfen von $\mathbf{A}^{-1}\mathbf{A} = \mathbf{I}$.

### Mini-Übung 2

1. Die Einheitsmatrix hat Einsen auf der Diagonale und Nullen sonst.
   Erzeugen Sie sie direkt mit `np.eye(3)` und vergleichen Sie sie mit
   dem Ergebnis von `A_inv @ A`. Stimmen sie überein?
2. Ein Café betreibt drei Filialen. In jeder Filiale gelten dieselben
   Preise (dieselbe Matrix `A`), aber die Tagesumsätze `b` sind
   verschieden. Erklären Sie in einem Satz, wie `np.linalg.solve` auch
   diesen Fall effizient löst.
3. Was gibt `np.linalg.inv(A_sing)` aus, wenn `A_sing` singulär ist?
   Testen Sie es.

In [ ]:
# Hier Ihren Code eingeben

### Mini-Übung 3

Lösen Sie das folgende Café-System vollständig: Legen Sie `A` und
`b` an, führen Sie den Determinanten-Test durch, lösen Sie mit `solve` und
sichern Sie das Ergebnis mit einer Probe ab.

| Tag | Kaffee | Tee | Eis | Umsatz (€) |
| --- | ------ | --- | ------ | ---------- |
| **Mo** | 5 | 2 | 3 | 25.60 |
| **Di** | 3 | 4 | 1 | 17.60 |
| **Mi** | 4 | 2 | 3 | 23.30 |

Beantworten Sie außerdem: Welcher Schritt aus diesem Kapitel könnte bei
einem größeren System mit 50 Unbekannten besonders lange dauern?

In [ ]:
# Hier Ihren Code eingeben

## Zusammenfassung und Ausblick

`np.linalg.solve(A, b)` löst ein LGS in einer Zeile, auch mit einer Matrix
`B` aus mehreren rechten Seiten. Die Probe `np.allclose(A @ x, b)` sichert
jedes Ergebnis ab. Die Inverse `np.linalg.inv(A)` berechnet
$\mathbf{A}^{-1}$ als Ergebnis und ist kein Ersatz für `solve`, sondern
sinnvoll, wenn die Inverse selbst weiterverwendet werden soll.

Im nächsten Kapitel verlassen wir den Obststand und wenden das Gelernte
auf ein reales Ingenieurproblem an: die Berechnung von Temperaturen in einer
mehrschichtigen Wand. Das Prinzip ist identisch, aber die Gleichungen kommen
jetzt aus der Physik.